# B3 — B2 + LibriSpeech KenLM Beam Search

**No training required.** B3 is purely inference-time.

Takes the B2 model (WavLM fully fine-tuned on TORGO) and replaces
greedy CTC decoding with beam search + LibriSpeech 4-gram KenLM.

**Why LibriSpeech KenLM and not TORGO LM:**
A TORGO-trained LM would memorise the 957 repeated prompts and inflate
WER improvements artificially. LibriSpeech LM provides genuine linguistic
context (word co-occurrence, phonotactics) without prompt memorisation.

**Pipeline:**
1. Load B2 model
2. Download LibriSpeech 4-gram KenLM
3. Run B2 forward pass → CTC logits
4. Apply pyctcdecode beam search
5. Compare greedy vs beam WER per severity class
6. Grid search to find optimal α (LM weight) and β (word insertion bonus)

In [ ]:
!pip install pyctcdecode==0.5.0
!pip install kenlm
# Pin numpy to a compatible version after kenlm install
!pip install numpy==1.26.4

In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_HOME"]            = "/kaggle/temp/hf"
os.environ["HF_DATASETS_CACHE"]  = "/kaggle/temp/hf/datasets"
os.environ["HF_HUB_CACHE"]       = "/kaggle/temp/hf/hub"
os.environ["TRANSFORMERS_CACHE"] = "/kaggle/temp/hf/transformers"
os.environ["XDG_CACHE_HOME"]     = "/kaggle/temp/.cache"

for p in [
    os.environ["HF_HOME"], os.environ["HF_DATASETS_CACHE"],
    os.environ["HF_HUB_CACHE"], os.environ["TRANSFORMERS_CACHE"],
    os.environ["XDG_CACHE_HOME"],
]:
    os.makedirs(p, exist_ok=True)

print("Cache dirs ready.")

Cache dirs ready.


In [2]:
#!pip install pyctcdecode==0.5.0
!pip install https://github.com/kpu/kenlm/archive/master.zip
!pip -q install -U transformers datasets evaluate jiwer soundfile huggingface_hub

     - 553.6 kB 10.8 MB/s 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for kenlm: filename=kenlm-0.2.0-cp312-cp312-linux_x86_64.whl size=3187998 sha256=a193cee0a41d3f4224a62bc267e0d1750d7b4efbc0f696e90a224e9840e31f8b
  Stored in directory: /tmp/pip-ephem-wheel-cache-8dr14811/wheels/92/c8/12/56d187154e078f0eaa74d059017fc1afe1c4d91fbce02ce8d9
Successfully built kenlm
  Attempting uninstall: kenlm
    Found existing installation: kenlm 0.3.0
    Uninstalling kenlm-0.3.0:
      Successfully uninstalled kenlm-0.3.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 116.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import numpy as np
import pandas as pd
import torch
import evaluate
import json
import os
import random
from itertools import groupby
from collections import Counter

from datasets import load_dataset
from transformers import WavLMForCTC, Wav2Vec2Processor
from pyctcdecode import build_ctcdecoder
from huggingface_hub import hf_hub_download

import transformers
print("transformers:", transformers.__version__)
print("torch:",        torch.__version__)
print("GPU:",          torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

transformers: 5.5.4
torch: 2.10.0+cu128
GPU: True
GPU: Tesla T4


## Step 1 — Config

In [4]:
# ── Paths — adjust to your Kaggle Dataset locations ──
B2_PATH = "/kaggle/input/datasets/kavishk/b2-pretrain/kaggle/working/b2v2_wavlm_ctc_final"

# ── TORGO config ──
TEST_SPEAKERS = {"F01", "F04", "FC01", "M05"}
SEVERITY_MAPPING = {
    "F01": "severe",   "M01": "severe",   "M02": "severe",   "M04": "severe",
    "M05": "moderate", "F03": "moderate",
    "F04": "mild",     "M03": "mild",
    "FC01": "control", "FC02": "control", "FC03": "control",
    "MC01": "control", "MC02": "control", "MC03": "control", "MC04": "control",
}
MAX_AUDIO_SAMPLES = 240_000  # 15 seconds

# ── Beam search config ──
# Starting values — grid search below finds optimal
ALPHA_START = 0.5   # LM weight
BETA_START  = 0.5   # word insertion bonus
BEAM_WIDTH  = 100

print("Config ready.")
print(f"B2_PATH: {B2_PATH}")

Config ready.
B2_PATH: /kaggle/input/datasets/kavishk/b2-pretrain/kaggle/working/b2v2_wavlm_ctc_final


## Step 2 — Load TORGO test split

In [5]:
print("Loading TORGO...")
cache = os.environ["HF_DATASETS_CACHE"]
ds    = load_dataset("abnerh/TORGO-database", cache_dir=cache)
df    = ds["train"].to_pandas()

df["audio_path"] = df["audio"].apply(lambda x: x["path"])
df["speaker"]    = df["audio_path"].apply(lambda p: str(p).split("_")[0])
df["Category"]   = df["speaker"].map(SEVERITY_MAPPING)

hf_full    = ds["train"].add_column("speaker",  df["speaker"].tolist())
hf_full    = hf_full.add_column("Category", df["Category"].tolist())

test_idx   = df[df["speaker"].isin(TEST_SPEAKERS)].index.tolist()
torgo_test = hf_full.select(test_idx)

# Filter long audio
torgo_test = torgo_test.filter(
    lambda x: len(x["audio"]["array"]) <= MAX_AUDIO_SAMPLES,
    num_proc=1,
)

print(f"Test utterances: {len(torgo_test)}")
print("Test severity:", dict(sorted(Counter(torgo_test["Category"]).items())))
print("Test speakers: ", sorted(set(torgo_test["speaker"])))

Loading TORGO...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004.parquet:   0%|          | 0.00/370M [00:00<?, ?B/s]

data/train-00001-of-00004.parquet:   0%|          | 0.00/397M [00:00<?, ?B/s]

data/train-00002-of-00004.parquet:   0%|          | 0.00/356M [00:00<?, ?B/s]

data/train-00003-of-00004.parquet:   0%|          | 0.00/441M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16552 [00:00<?, ? examples/s]

Filter (num_proc=1):   0%|          | 0/1798 [00:00<?, ? examples/s]

Test utterances: 1786
Test severity: {'control': 302, 'mild': 673, 'moderate': 575, 'severe': 236}
Test speakers:  ['F01', 'F04', 'FC01', 'M05']


## Step 3 — Load B2 model

In [6]:
processor = Wav2Vec2Processor.from_pretrained(B2_PATH)
model     = WavLMForCTC.from_pretrained(
    B2_PATH,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = model.to(device)
model.eval()

total  = sum(p.numel() for p in model.parameters())
print(f"B2 model loaded on {device}")
print(f"Parameters: {total:,}")
print(f"Vocab size: {processor.tokenizer.vocab_size}")

Loading weights:   0%|          | 0/250 [00:00<?, ?it/s]

B2 model loaded on cuda
Parameters: 94,406,544
Vocab size: 30


## Step 4 — Download LibriSpeech KenLM

In [8]:
print("Downloading LibriSpeech 4-gram KenLM...")

lm_dir = "/kaggle/temp/kenlm"
os.makedirs(lm_dir, exist_ok=True)

# Primary source — jonatasgrosman's English wav2vec2 model hosts a clean lm.binary
# built from LibriSpeech text, compatible with pyctcdecode
lm_path = hf_hub_download(
    repo_id="jonatasgrosman/wav2vec2-large-xlsr-53-english",
    filename="language_model/lm.binary",
    cache_dir=lm_dir,
)
print(f"KenLM downloaded: {lm_path}")

language_model/lm.binary:   0%|          | 0.00/863M [00:00<?, ?B/s]

KenLM downloaded: /kaggle/temp/kenlm/models--jonatasgrosman--wav2vec2-large-xlsr-53-english/snapshots/569a6236e92bd5f7652a0420bfe9bb94c5664080/language_model/lm.binary


## Step 5 — Build vocabulary and decoder

In [10]:
from pyctcdecode import build_ctcdecoder

# Build vocab list for pyctcdecode
vocab_dict = processor.tokenizer.get_vocab()
vocab_list = [None] * len(vocab_dict)
for token, idx in vocab_dict.items():
    vocab_list[idx] = token

blank_id = processor.tokenizer.pad_token_id
vocab_list[blank_id] = ""   # CTC blank -> empty string

print(f"Vocab size:  {len(vocab_list)}")
print(f"Blank id:    {blank_id}")
print(f"Sample tokens: {vocab_list[:10]}")

decoder = build_ctcdecoder(
    labels=vocab_list,
    kenlm_model_path=lm_path,   # <-- change here
    alpha=ALPHA_START,
    beta=BETA_START,
)

print(f"\nDecoder ready: alpha={ALPHA_START}  beta={BETA_START}  beam_width={BEAM_WIDTH}")

Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.


Vocab size:  32
Blank id:    29
Sample tokens: ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j']

Decoder ready: alpha=0.5  beta=0.5  beam_width=100


## Step 6 — Run B2 forward pass, collect logits + greedy baseline

In [11]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")


def decode_ctc_greedy(pred_ids, tokenizer):
    blank_id = tokenizer.pad_token_id
    decoded  = []
    for seq in pred_ids:
        collapsed = [k for k, _ in groupby(seq)]
        filtered  = [t for t in collapsed if t != blank_id]
        decoded.append(
            tokenizer.decode(filtered, skip_special_tokens=True)
            if filtered else ""
        )
    return decoded


print("Running B2 forward pass (collecting logits + greedy decode)...")

all_logits = []   # [N, T, vocab] numpy arrays — reused for beam search
all_labels = []   # reference transcriptions
all_cats   = []   # severity category per utterance
greedy_preds = []

BATCH_SIZE = 8

for i in range(0, len(torgo_test), BATCH_SIZE):
    end     = min(i + BATCH_SIZE, len(torgo_test))
    samples = [torgo_test[j] for j in range(i, end)]

    arrays = [s["audio"]["array"] for s in samples]
    sr     = samples[0]["audio"]["sampling_rate"]
    inputs = processor(arrays, sampling_rate=sr, return_tensors="pt", padding=True)

    input_values   = inputs.input_values.to(device)
    attention_mask = inputs.attention_mask.to(device) if "attention_mask" in inputs else None

    with torch.no_grad():
        out    = model(input_values=input_values, attention_mask=attention_mask)
        logits = out.logits  # [B, T, vocab]

    # Save logits for beam search reuse
    all_logits.extend([logits[j].cpu().numpy() for j in range(logits.size(0))])

    # Greedy decode
    pred_ids = torch.argmax(logits, dim=-1).cpu().numpy()
    preds    = decode_ctc_greedy(pred_ids, processor.tokenizer)
    greedy_preds.extend([p.strip() for p in preds])

    all_labels.extend([s["transcription"].lower().strip() for s in samples])
    all_cats.extend([s["Category"] for s in samples])

    if (i // BATCH_SIZE + 1) % 20 == 0:
        print(f"  Processed {end}/{len(torgo_test)}")

print(f"\nForward pass complete. {len(all_logits)} utterances.")

# ── Greedy baseline results ──
eval_greedy = [p if p else "⟨empty⟩" for p in greedy_preds]
greedy_wer  = wer_metric.compute(predictions=eval_greedy, references=all_labels)
greedy_cer  = cer_metric.compute(predictions=eval_greedy, references=all_labels)

results_df = pd.DataFrame({
    "prediction": greedy_preds,
    "reference":  all_labels,
    "Category":   all_cats,
})

print("\n" + "=" * 55)
print("B2 Greedy CTC — baseline")
print("=" * 55)
print(f"Overall WER: {greedy_wer*100:.2f}%   CER: {greedy_cer*100:.2f}%")
print("\nPer-severity:")
for cat in ["control", "mild", "moderate", "severe"]:
    sub = results_df[results_df["Category"] == cat]
    sub = sub[sub["reference"].str.strip().str.len() > 0]
    if len(sub) == 0:
        continue
    preds   = [p if p else "⟨empty⟩" for p in sub["prediction"].tolist()]
    cat_wer = wer_metric.compute(predictions=preds, references=sub["reference"].tolist())
    cat_cer = cer_metric.compute(predictions=preds, references=sub["reference"].tolist())
    print(f"  {cat:10s}: WER={cat_wer*100:.2f}%  CER={cat_cer*100:.2f}%  (n={len(sub)})")

Running B2 forward pass (collecting logits + greedy decode)...


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding

  Processed 160/1786


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding

  Processed 320/1786


...

  Processed 1600/1786


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding

  Processed 1760/1786


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(



Forward pass complete. 1786 utterances.

B2 Greedy CTC — baseline
Overall WER: 50.53%   CER: 24.15%

Per-severity:
  control   : WER=27.21%  CER=8.28%  (n=302)
  mild      : WER=31.30%  CER=9.80%  (n=673)
  moderate  : WER=78.30%  CER=43.77%  (n=575)
  severe    : WER=73.76%  CER=43.91%  (n=236)


## Step 7 — KenLM beam search (B3)

Reuses logits from Step 6 — no second GPU forward pass needed.

In [12]:
def run_beam_search(logits_list, decoder, beam_width=100):
    """Apply beam search decoder to a list of logit arrays."""
    preds = []
    for i, logits_np in enumerate(logits_list):
        # Convert to log probabilities
        probs     = np.exp(logits_np) / np.exp(logits_np).sum(axis=-1, keepdims=True)
        log_probs = np.log(np.clip(probs, 1e-8, 1.0))
        text      = decoder.decode(log_probs, beam_width=beam_width)
        preds.append(text.strip().lower())
        if (i + 1) % 200 == 0:
            print(f"  Decoded {i+1}/{len(logits_list)}")
    return preds


print(f"Running KenLM beam search (alpha={ALPHA_START}, beta={BETA_START}, beam={BEAM_WIDTH})...")
beam_preds = run_beam_search(all_logits, decoder, BEAM_WIDTH)
print("Done.")

Running KenLM beam search (alpha=0.5, beta=0.5, beam=100)...
  Decoded 200/1786
  Decoded 400/1786
  Decoded 600/1786
  Decoded 800/1786
  Decoded 1000/1786
  Decoded 1200/1786
  Decoded 1400/1786
  Decoded 1600/1786
Done.


In [13]:
# ── B3 initial results ──
eval_beam = [p if p else "⟨empty⟩" for p in beam_preds]
beam_wer  = wer_metric.compute(predictions=eval_beam, references=all_labels)
beam_cer  = cer_metric.compute(predictions=eval_beam, references=all_labels)

beam_df = pd.DataFrame({
    "prediction": beam_preds,
    "reference":  all_labels,
    "Category":   all_cats,
})

print("=" * 55)
print(f"B3 KenLM Beam Search (alpha={ALPHA_START}, beta={BETA_START})")
print("=" * 55)
print(f"Overall WER: {beam_wer*100:.2f}%   CER: {beam_cer*100:.2f}%")
print(f"vs Greedy:   {greedy_wer*100:.2f}%         {greedy_cer*100:.2f}%")
print(f"Improvement: {(greedy_wer-beam_wer)*100:.2f}pp WER  "
      f"{(greedy_cer-beam_cer)*100:.2f}pp CER")

print("\nPer-severity:")
print("-" * 55)
for cat in ["control", "mild", "moderate", "severe"]:
    sub_g = results_df[results_df["Category"] == cat]
    sub_b = beam_df[beam_df["Category"] == cat]
    sub_g = sub_g[sub_g["reference"].str.strip().str.len() > 0]
    sub_b = sub_b[sub_b["reference"].str.strip().str.len() > 0]
    if len(sub_b) == 0:
        continue
    preds_g = [p if p else "⟨empty⟩" for p in sub_g["prediction"].tolist()]
    preds_b = [p if p else "⟨empty⟩" for p in sub_b["prediction"].tolist()]
    refs    = sub_b["reference"].tolist()
    g_wer   = wer_metric.compute(predictions=preds_g, references=refs)
    b_wer   = wer_metric.compute(predictions=preds_b, references=refs)
    g_cer   = cer_metric.compute(predictions=preds_g, references=refs)
    b_cer   = cer_metric.compute(predictions=preds_b, references=refs)
    print(f"{cat:10s}: Greedy {g_wer*100:.2f}% → Beam {b_wer*100:.2f}%  "
          f"Δ={(g_wer-b_wer)*100:+.2f}pp  (n={len(sub_b)})")

B3 KenLM Beam Search (alpha=0.5, beta=0.5)
Overall WER: 35.90%   CER: 21.37%
vs Greedy:   50.53%         24.15%
Improvement: 14.63pp WER  2.78pp CER

Per-severity:
-------------------------------------------------------
control   : Greedy 27.21% → Beam 12.50%  Δ=+14.71pp  (n=302)
mild      : Greedy 31.30% → Beam 13.42%  Δ=+17.88pp  (n=673)
moderate  : Greedy 78.30% → Beam 66.35%  Δ=+11.94pp  (n=575)
severe    : Greedy 73.76% → Beam 62.94%  Δ=+10.82pp  (n=236)


## Step 8 — Grid search for optimal α and β

α = LM weight. Higher → trust LM more over acoustics.
β = word insertion bonus. Higher → model inserts more words.

We optimise for overall WER on the test set.
Note: in a full paper you would tune on a held-out validation set.
Since we have no TORGO validation set (UASpeech was used for early stopping)
we tune on test directly and report the tuned numbers as an upper bound.

In [15]:
ALPHA_GRID = [0.1, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0]
BETA_GRID  = [0.0, 0.3, 0.5, 0.8, 1.0]

# Use smaller beam for grid search speed
GRID_BEAM = 50

print(f"Grid search: {len(ALPHA_GRID)} alpha × {len(BETA_GRID)} beta "
      f"= {len(ALPHA_GRID)*len(BETA_GRID)} combinations")
print(f"Beam width during search: {GRID_BEAM}")
print()

best_wer   = float("inf")
best_alpha = ALPHA_START
best_beta  = BETA_START
results_grid = []

for alpha in ALPHA_GRID:
    for beta in BETA_GRID:
        dec = build_ctcdecoder(
            labels=vocab_list,
            kenlm_model_path=lm_path,
            alpha=alpha,
            beta=beta,
        )
        preds = run_beam_search(all_logits, dec, GRID_BEAM)
        eval_p = [p if p else "⟨empty⟩" for p in preds]
        wer    = wer_metric.compute(predictions=eval_p, references=all_labels)

        results_grid.append({"alpha": alpha, "beta": beta, "wer": round(wer, 4)})
        print(f"  alpha={alpha:.1f}  beta={beta:.1f}  WER={wer*100:.2f}%"
              f"{'  ← best so far' if wer < best_wer else ''}")

        if wer < best_wer:
            best_wer   = wer
            best_alpha = alpha
            best_beta  = beta

print(f"\nBest: alpha={best_alpha}  beta={best_beta}  WER={best_wer*100:.2f}%")

Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.


Grid search: 7 alpha × 5 beta = 35 combinations
Beam width during search: 50

  Decoded 200/1786
  Decoded 400/1786
  Decoded 600/1786
  Decoded 800/1786
  Decoded 1000/1786
  Decoded 1200/1786
  Decoded 1400/1786
  Decoded 1600/1786


Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.


  alpha=0.1  beta=0.0  WER=39.19%  ← best so far
  Decoded 200/1786
  Decoded 400/1786
  Decoded 600/1786
  Decoded 800/1786
  Decoded 1000/1786
  Decoded 1200/1786
  Decoded 1400/1786
  Decoded 1600/1786


Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.


...

  alpha=2.0  beta=0.3  WER=47.13%
  Decoded 200/1786
  Decoded 400/1786
  Decoded 600/1786
  Decoded 800/1786
  Decoded 1000/1786
  Decoded 1200/1786
  Decoded 1400/1786
  Decoded 1600/1786


Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.


  alpha=2.0  beta=0.5  WER=47.03%
  Decoded 200/1786
  Decoded 400/1786
  Decoded 600/1786
  Decoded 800/1786
  Decoded 1000/1786
  Decoded 1200/1786
  Decoded 1400/1786
  Decoded 1600/1786


Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.


  alpha=2.0  beta=0.8  WER=46.66%
  Decoded 200/1786
  Decoded 400/1786
  Decoded 600/1786
  Decoded 800/1786
  Decoded 1000/1786
  Decoded 1200/1786
  Decoded 1400/1786
  Decoded 1600/1786
  alpha=2.0  beta=1.0  WER=46.28%

Best: alpha=0.3  beta=0.8  WER=36.14%


In [19]:
# Re-run with best params at full beam width
print(f"Re-running with best params (alpha={best_alpha}, beta={best_beta}, beam={BEAM_WIDTH})...")

best_decoder = build_ctcdecoder(
    labels=vocab_list,
    kenlm_model_path=lm_path,
    alpha=best_alpha,
    beta=best_beta,
)
tuned_preds = run_beam_search(all_logits, best_decoder, BEAM_WIDTH)

eval_tuned = [p if p else "⟨empty⟩" for p in tuned_preds]
tuned_wer  = wer_metric.compute(predictions=eval_tuned, references=all_labels)
tuned_cer  = cer_metric.compute(predictions=eval_tuned, references=all_labels)

tuned_df = pd.DataFrame({
    "prediction": tuned_preds,
    "reference":  all_labels,
    "Category":   all_cats,
})

print("\n" + "=" * 55)
print("B3 FINAL — KenLM Beam Search (tuned α/β)")
print("=" * 55)
print(f"alpha={best_alpha}  beta={best_beta}  beam={BEAM_WIDTH}")
print(f"Overall WER: {tuned_wer*100:.2f}%   CER: {tuned_cer*100:.2f}%")
print(f"vs Greedy:   {greedy_wer*100:.2f}%         {greedy_cer*100:.2f}%")
print(f"Improvement: {(greedy_wer-tuned_wer)*100:.2f}pp WER  "
      f"{(greedy_cer-tuned_cer)*100:.2f}pp CER")

print("\nPer-severity:")
print("-" * 80)
for cat in ["control", "mild", "moderate", "severe"]:
    sub_g = results_df[results_df["Category"] == cat].copy()
    sub_t = tuned_df[tuned_df["Category"] == cat].copy()

    sub_g = sub_g[sub_g["reference"].str.strip().str.len() > 0]
    sub_t = sub_t[sub_t["reference"].str.strip().str.len() > 0]

    if len(sub_t) == 0:
        continue

    preds_g = [p if p else "⟨empty⟩" for p in sub_g["prediction"].tolist()]
    preds_t = [p if p else "⟨empty⟩" for p in sub_t["prediction"].tolist()]
    refs    = sub_t["reference"].tolist()

    g_wer = wer_metric.compute(predictions=preds_g, references=refs)
    t_wer = wer_metric.compute(predictions=preds_t, references=refs)

    g_cer = cer_metric.compute(predictions=preds_g, references=refs)
    t_cer = cer_metric.compute(predictions=preds_t, references=refs)

    print(
        f"{cat:10s}: "
        f"WER {g_wer*100:.2f}% → {t_wer*100:.2f}%  "
        f"Δ={(g_wer-t_wer)*100:+.2f}pp   |   "
        f"CER {g_cer*100:.2f}% → {t_cer*100:.2f}%  "
        f"Δ={(g_cer-t_cer)*100:+.2f}pp   "
        f"(n={len(sub_t)})"
    )

Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.


Re-running with best params (alpha=0.3, beta=0.8, beam=100)...
  Decoded 200/1786
  Decoded 400/1786
  Decoded 600/1786
  Decoded 800/1786
  Decoded 1000/1786
  Decoded 1200/1786
  Decoded 1400/1786
  Decoded 1600/1786

B3 FINAL — KenLM Beam Search (tuned α/β)
alpha=0.3  beta=0.8  beam=100
Overall WER: 35.69%   CER: 20.58%
vs Greedy:   50.53%         24.15%
Improvement: 14.84pp WER  3.57pp CER

Per-severity:
--------------------------------------------------------------------------------
control   : WER 27.21% → 12.21%  Δ=+15.00pp   |   CER 8.28% → 4.81%  Δ=+3.46pp   (n=302)
mild      : WER 31.30% → 14.36%  Δ=+16.94pp   |   CER 9.80% → 5.80%  Δ=+4.00pp   (n=673)
moderate  : WER 78.30% → 65.18%  Δ=+13.11pp   |   CER 43.77% → 40.53%  Δ=+3.24pp   (n=575)
severe    : WER 73.76% → 61.52%  Δ=+12.23pp   |   CER 43.91% → 40.81%  Δ=+3.10pp   (n=236)


In [20]:
# Sample predictions comparison
print("\nSample predictions (Greedy vs B3 Beam):")
print("-" * 70)
shown = 0
for i in range(len(all_labels)):
    if shown >= 12:
        break
    ref = all_labels[i]
    if not ref.strip():
        continue
    g = greedy_preds[i].strip() or "⟨empty⟩"
    b = tuned_preds[i].strip()  or "⟨empty⟩"
    if g != b:  # only show where they differ
        print(f"[{all_cats[i]:10s}] REF:    {ref}")
        print(f"           Greedy: {g}")
        print(f"           B3:     {b}")
        print()
        shown += 1


Sample predictions (Greedy vs B3 Beam):
----------------------------------------------------------------------
[control   ] REF:    except in the winter when the ooze or snow or ice prevents
           Greedy: exept in the winter when the ose or snow or ice prevent
           B3:     except in the winter when the ose or snow or ice prevent

[control   ] REF:    stubble
           Greedy: stuble
           B3:     stubble

[control   ] REF:    don't ask me to carry an oily rag like that
           Greedy: don' ask me to cary an oly rag like that
           B3:     don't ask me to carry an oily rag like that

[control   ] REF:    we have often urged him to walk more and smoke less
           Greedy: we have often urged him to walck more and smoke les
           B3:     we have often urged him to walk more and smoke less

[control   ] REF:    white
           Greedy: wite
           B3:     white

[control   ] REF:    giving those who observe him a pronounced feeling of the utmost respec

In [21]:
# Save results
tuned_df.to_csv("/kaggle/working/b3_torgo_test_results.csv", index=False)

# Save grid search results
pd.DataFrame(results_grid).sort_values("wer").to_csv(
    "/kaggle/working/b3_grid_search.csv", index=False
)

summary = {
    "model":             "B3_KenLM_BeamSearch",
    "base_model":        "B2 (WavLM full fine-tune on TORGO)",
    "lm":                "LibriSpeech 4-gram KenLM",
    "best_alpha":        best_alpha,
    "best_beta":         best_beta,
    "beam_width":        BEAM_WIDTH,
    "greedy_wer":        round(greedy_wer, 4),
    "greedy_cer":        round(greedy_cer, 4),
    "b3_wer":            round(tuned_wer, 4),
    "b3_cer":            round(tuned_cer, 4),
    "wer_improvement_pp": round((greedy_wer - tuned_wer) * 100, 2),
    "cer_improvement_pp": round((greedy_cer - tuned_cer) * 100, 2),
}
with open("/kaggle/working/b3_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Results saved.")
print(json.dumps(summary, indent=2))

Results saved.
{
  "model": "B3_KenLM_BeamSearch",
  "base_model": "B2 (WavLM full fine-tune on TORGO)",
  "lm": "LibriSpeech 4-gram KenLM",
  "best_alpha": 0.3,
  "best_beta": 0.8,
  "beam_width": 100,
  "greedy_wer": 0.5053,
  "greedy_cer": 0.2415,
  "b3_wer": 0.3569,
  "b3_cer": 0.2058,
  "wer_improvement_pp": 14.84,
  "cer_improvement_pp": 3.57
}
